In [1]:
import pandas as pd

In [2]:
data = pd.read_csv(r"C:\Users\AAKAR\OneDrive\Documents\ECOMMERCE_RECOMMENDATION\flipkart_product_review.csv")
data.head()

,product_id,product_title,rating,summary,review
0,ACCFZGAQJGYCYDCM,BoAt Rockerz 235v2 with ASAP charging Version ...,5,Terrific purchase,1-more flexible2-bass is very high3-sound clar...
1,ACCFZGAQJGYCYDCM,BoAt Rockerz 235v2 with ASAP charging Version ...,5,Terrific purchase,Super sound and good looking I like that prize
2,ACCFZGAQJGYCYDCM,BoAt Rockerz 235v2 with ASAP charging Version ...,5,Super!,Very much satisfied with the device at this pr...
3,ACCFZGAQJGYCYDCM,BoAt Rockerz 235v2 with ASAP charging Version ...,5,Super!,"Nice headphone, bass was very good and sound i..."
4,ACCFZGAQJGYCYDCM,BoAt Rockerz 235v2 with ASAP charging Version ...,5,Terrific purchase,Sound quality super battery backup super quali...


In [3]:
import sys
!{sys.executable} -m pip install -q langchain langchain-community langchain-astradb langchain-groq pypdf

In [4]:
data.head()

,product_id,product_title,rating,summary,review
0,ACCFZGAQJGYCYDCM,BoAt Rockerz 235v2 with ASAP charging Version ...,5,Terrific purchase,1-more flexible2-bass is very high3-sound clar...
1,ACCFZGAQJGYCYDCM,BoAt Rockerz 235v2 with ASAP charging Version ...,5,Terrific purchase,Super sound and good looking I like that prize
2,ACCFZGAQJGYCYDCM,BoAt Rockerz 235v2 with ASAP charging Version ...,5,Super!,Very much satisfied with the device at this pr...
3,ACCFZGAQJGYCYDCM,BoAt Rockerz 235v2 with ASAP charging Version ...,5,Super!,"Nice headphone, bass was very good and sound i..."
4,ACCFZGAQJGYCYDCM,BoAt Rockerz 235v2 with ASAP charging Version ...,5,Terrific purchase,Sound quality super battery backup super quali...


In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 450 entries, 0 to 449
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   product_id     450 non-null    object
 1   product_title  450 non-null    object
 2   rating         450 non-null    int64 
 3   summary        450 non-null    object
 4   review         450 non-null    object
dtypes: int64(1), object(4)
memory usage: 17.7+ KB


In [6]:
data = data[["product_title", "review"]]
data.head()

,product_title,review
0,BoAt Rockerz 235v2 with ASAP charging Version ...,1-more flexible2-bass is very high3-sound clar...
1,BoAt Rockerz 235v2 with ASAP charging Version ...,Super sound and good looking I like that prize
2,BoAt Rockerz 235v2 with ASAP charging Version ...,Very much satisfied with the device at this pr...
3,BoAt Rockerz 235v2 with ASAP charging Version ...,"Nice headphone, bass was very good and sound i..."
4,BoAt Rockerz 235v2 with ASAP charging Version ...,Sound quality super battery backup super quali...


Converting data into a document

In [7]:
product_list = []
for index, row in data.iterrows():
    object = {
        "product_name" : row["product_title"],
        "review" : row["review"]
    }
    product_list.append(object)

In [8]:
product_list[0]

{'product_name': 'BoAt Rockerz 235v2 with ASAP charging Version 5.0 Bluetooth Headset',
 'review': "1-more flexible2-bass is very high3-sound clarity is good 4-battery back up to 6 to 8 hour's 5-main thing is fastest charging system is available in that. Only 20 min charge and get long up to 4 hours back up 6-killing look awesome 7-for gaming that product does not support 100% if you want for gaming then I'll recommend you please don't buy but you want for only music then this product is very well for you.. 8-no more wireless headphones are comparing with that headphones at this pric..."}

In [9]:
from langchain_core.documents import Document

In [10]:
docs = []
for object in product_list:
    metadata = {"product_name" : object["product_name"]}
    page_content = object["review"]
    
    doc = Document(page_content = page_content, metadata = metadata)
    docs.append(doc)

In [11]:
docs[0]

Document(metadata={'product_name': 'BoAt Rockerz 235v2 with ASAP charging Version 5.0 Bluetooth Headset'}, page_content="1-more flexible2-bass is very high3-sound clarity is good 4-battery back up to 6 to 8 hour's 5-main thing is fastest charging system is available in that. Only 20 min charge and get long up to 4 hours back up 6-killing look awesome 7-for gaming that product does not support 100% if you want for gaming then I'll recommend you please don't buy but you want for only music then this product is very well for you.. 8-no more wireless headphones are comparing with that headphones at this pric...")

In [12]:
len(docs)

450

In [ ]:
GROQ_API = ""
ASTRA_DB_API_ENDPOINT = ""
ASTRA_DB_APPLICATION_TOKEN = ""
ASTRA_DB_KEYSPACE = ""
HF_TOKEN = ""

In [14]:
from langchain_huggingface import HuggingFaceEndpointEmbeddings

In [20]:
embeddings = HuggingFaceEndpointEmbeddings(
    huggingfacehub_api_token = HF_TOKEN,
    model = "BAAI/bge-base-en-v1.5"
)

In [21]:
from langchain_astradb import AstraDBVectorStore

In [22]:
vstore = AstraDBVectorStore(
    embedding = embeddings,
    collection_name = "PYTHONESH_ECOMMERCE",
    api_endpoint = ASTRA_DB_API_ENDPOINT,
    token = ASTRA_DB_APPLICATION_TOKEN,
    namespace = ASTRA_DB_KEYSPACE
)   

In [23]:
insert_ids = vstore.add_documents(docs)

In [24]:
from langchain_groq import ChatGroq

In [25]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [26]:
model = ChatGroq(groq_api_key=GROQ_API, model="openai/gpt-oss-120b", temperature=0.5)

In [27]:
retriever_prompt = ("Given a chat history and the latest user question which might reference context in the chat history,"
    "formulate a standalone question which can be understood without the chat history."
    "Do NOT answer the question, just reformulate it if needed and otherwise return it as is."
    )

In [28]:
retriever = vstore.as_retriever(search_kwargs={"k": 16})

In [29]:
from langchain_core.prompts import ChatPromptTemplate

In [30]:
contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", retriever_prompt),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}")
    ]
)

In [31]:
PRODUCT_BOT_TEMPLATE = """
    Your ecommercebot bot is an expert in product recommendations and customer queries.
    It analyzes product titles and reviews to provide accurate and helpful responses.
    Ensure your answers are relevant to the product context and refrain from straying off-topic.
    Your responses should be concise and informative.

    CONTEXT:
    {context}

    QUESTION: {input}

    YOUR ANSWER:

    """

In [32]:
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", PRODUCT_BOT_TEMPLATE),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}")
    ]
)

In [33]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def get_search_query(inputs):
    if not inputs.get("chat_history"):
        return inputs["input"]
    rephrase_chain = contextualize_q_prompt | model | StrOutputParser()
    return rephrase_chain.invoke(inputs)

qa_chain = qa_prompt | model | StrOutputParser()

In [34]:
def full_chain(inputs):
    query = get_search_query(inputs)
    docs = retriever.invoke(query)
    answer = qa_chain.invoke({
        "context" : format_docs(docs),
        "input" : inputs["input"],
        "chat_history" : inputs.get("chat_history", [])
    })
    return {"answer" : answer, "context" : docs}

chain = RunnableLambda(full_chain)

In [35]:
chat_history = []

In [36]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

C:\Users\AAKAR\AppData\Local\Temp\ipykernel_10260\1529080270.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory


In [37]:
store = {}

In [38]:
def get_session_history(session_id : str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

In [39]:
chain_with_memory = RunnableWithMessageHistory(
    chain, 
    get_session_history,
    input_messages_key = "input",
    history_messages_key = "chat_history",
    output_messages_key = "answer",
)

D:\ProgramData\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3699: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [40]:
chain_with_memory.invoke(
    {"input": "can you tell me the best bluetooth buds ? "},
    config = {
        "configurable" : {"session_id" : "test"}
    }
)["answer"]

'**Top Recommendation – Realme Buds (Bluetooth)**  \n\n| Feature | Why It Stands Out |\n|---------|-------------------|\n| **Sound & Bass** | Consistently praised for “astonishing” bass and clear mids/treble. Reviewers say you can hear lyrics even in a crowd. |\n| **Battery Life** | 10‑14\u202fhours of continuous music, plus up to 7\u202fdays standby. Fast charge: ~1\u202fh\u202f20\u202fmin for a full charge. |\n| **Fit & Build** | Flexible velvet‑type ear‑tips, lightweight metal filters, and a smooth, premium‑feel design. |\n| **Connectivity** | Bluetooth pairing in ~3\u202fseconds, auto‑reconnect within 2‑4\u202fseconds, stable 75\u202fft range. |\n| **Water Resistance** | Sweat‑proof (suitable for gym use). |\n| **Value** | Budget‑friendly price, often rated 5‑star for price‑to‑performance ratio. |\n| **Extras** | Low‑latency mode (good for gaming), clear mic quality, magnetic on/off switch on some models. |\n\n**Why Realme beats the competition in this price segment**\n\n- **OnePlu

In [41]:
store

{'test': InMemoryChatMessageHistory(messages=[HumanMessage(content='can you tell me the best bluetooth buds ? ', additional_kwargs={}, response_metadata={}), AIMessage(content='**Top Recommendation – Realme Buds (Bluetooth)**  \n\n| Feature | Why It Stands Out |\n|---------|-------------------|\n| **Sound & Bass** | Consistently praised for “astonishing” bass and clear mids/treble. Reviewers say you can hear lyrics even in a crowd. |\n| **Battery Life** | 10‑14\u202fhours of continuous music, plus up to 7\u202fdays standby. Fast charge: ~1\u202fh\u202f20\u202fmin for a full charge. |\n| **Fit & Build** | Flexible velvet‑type ear‑tips, lightweight metal filters, and a smooth, premium‑feel design. |\n| **Connectivity** | Bluetooth pairing in ~3\u202fseconds, auto‑reconnect within 2‑4\u202fseconds, stable 75\u202fft range. |\n| **Water Resistance** | Sweat‑proof (suitable for gym use). |\n| **Value** | Budget‑friendly price, often rated 5‑star for price‑to‑performance ratio. |\n| **Extras

In [42]:
chain_with_memory.invoke(
   {"input": "what was my previous question?"},
    config={
        "configurable": {"session_id": "test"}
    },  
)["answer"]

'Your previous question was:\n\n**“Can you tell me the best Bluetooth buds?”**'

In [46]:
store

{'test': InMemoryChatMessageHistory(messages=[HumanMessage(content='can you tell me the best bluetooth buds ? ', additional_kwargs={}, response_metadata={}), AIMessage(content='**Top Recommendation – Realme Buds (Bluetooth)**  \n\n| Feature | Why It Stands Out |\n|---------|-------------------|\n| **Sound & Bass** | Consistently praised for “astonishing” bass and clear mids/treble. Reviewers say you can hear lyrics even in a crowd. |\n| **Battery Life** | 10‑14\u202fhours of continuous music, plus up to 7\u202fdays standby. Fast charge: ~1\u202fh\u202f20\u202fmin for a full charge. |\n| **Fit & Build** | Flexible velvet‑type ear‑tips, lightweight metal filters, and a smooth, premium‑feel design. |\n| **Connectivity** | Bluetooth pairing in ~3\u202fseconds, auto‑reconnect within 2‑4\u202fseconds, stable 75\u202fft range. |\n| **Water Resistance** | Sweat‑proof (suitable for gym use). |\n| **Value** | Budget‑friendly price, often rated 5‑star for price‑to‑performance ratio. |\n| **Extras